<a href="https://colab.research.google.com/github/syedvasimfathima/Conversation-Q-A/blob/main/Conversational_Q%26A_Chat_with_History_Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-openai streamlit pyngrok

In [ ]:
#Secured my API Key in Google colab
#api key is My_First_LLM (from OpenRouter)
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('API')


In [ ]:
%%writefile app.py

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
import streamlit as st
import time # Added for sleep
import os


st.set_page_config(page_title='Q & A Demo')
st.title("Q&A Chat Bot")

# Initialize chat history in session state
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    if isinstance(message, HumanMessage):
        with st.chat_message("user"):
            st.markdown(message.content)
    elif isinstance(message, AIMessage):
        with st.chat_message("assistant"):
            st.markdown(message.content)

# Initialize llm (assuming OPENAI_API_KEY is set in environment)
llm = ChatOpenAI(model="gpt-3.5-turbo", openai_api_key=os.environ.get("OPENAI_API_KEY"), base_url="https://openrouter.ai/api/v1")

# Define the prompt template
promt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a nice AI bot"),
        ("human", "{query}")
    ]
)

# Accept user input
if query := st.chat_input("Ask me anything... (type 'exit' to clear history)"):
    # Add user message to chat history
    st.session_state.messages.append(HumanMessage(content=query))
    with st.chat_message("user"):
        st.markdown(query)

    # Check for exit command
    if query.lower() == 'exit':
        st.session_state.messages = [] # Clear history
        #st.experimental_rerun() # Rerun to reflect cleared history
        st.subheader("Please refresh the page or enter anything for page refresh")
    else:
        # Create a simple chain for current query
        chain = promt | llm

        # Invoke the LLM
        response = chain.invoke(
            {"query": query},
            config={
                "configurable": {"session_id": "any_session_id"}
            }
        )

        # Add AI response to chat history
        st.session_state.messages.append(AIMessage(content=response.content))
        with st.chat_message("assistant"):
            st.markdown(response.content)

Overwriting app.py


In [ ]:
import time

# Kill any processes running on port 8501 (Streamlit's default port)
!fuser -k 8501/tcp

# Run the Streamlit app in the background
# Output is redirected to /dev/null to keep the notebook clean
!streamlit run app.py &>/dev/null&

print("Streamlit app is starting...")
time.sleep(5) # Give Streamlit some time to boot up
print("Streamlit app should be running.")

Streamlit app is starting...
Streamlit app should be running.


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Make sure to store your ngrok authtoken in Colab's secrets as 'NGROK_AUTH_TOKEN'
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(ngrok_auth_token)

# Kill any existing ngrok tunnels to avoid conflicts
ngrok.kill()

print("Connecting ngrok...")
public_url = ngrok.connect(8501)

print(f"Streamlit App Public URL: {public_url}")

Connecting ngrok...
Streamlit App Public URL: NgrokTunnel: "https://turbulent-junkman-despise.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
import subprocess

# Kill all ngrok processes
try:
    subprocess.run(['pkill', 'ngrok'], check=True, capture_output=True)
    print('All ngrok processes have been terminated.')
except subprocess.CalledProcessError as e:
    print(f'Error terminating ngrok processes: {e.stderr.decode()}')
    print('This might happen if no ngrok processes were running.')

All ngrok processes have been terminated.


In [ ]:
# Verify that ngrok processes are no longer running
import subprocess

try:
    result = subprocess.run(['ps', 'aux'], check=True, capture_output=True, text=True)
    ngrok_processes = [line for line in result.stdout.splitlines() if 'ngrok' in line and 'grep' not in line]
    if ngrok_processes:
        print('Found active ngrok processes:')
        for p in ngrok_processes:
            print(p)
    else:
        print('No active ngrok processes found.')
except subprocess.CalledProcessError as e:
    print(f'Error listing processes: {e.stderr.decode()}')


No active ngrok processes found.
